In [10]:
import torch
import re
import json
from PIL import Image
from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [12]:
# Load Surya
foundation_predictor = FoundationPredictor()
recognition_predictor = RecognitionPredictor(foundation_predictor)
detection_predictor = DetectionPredictor()
print("Surya loaded.")


Surya loaded.


In [13]:
# OCR the same Wikipedia test image
image_path = "../data/inputs/test_arabic_01.png"   # adjust to your actual path
image = Image.open(image_path).convert("RGB")

predictions = recognition_predictor([image], det_predictor=detection_predictor)
text_lines = predictions[0].text_lines

arabic_lines = [line.text for line in text_lines]
line_confidences = [line.confidence for line in text_lines]

Recognizing Text: 100%|█████████████████████████████████| 8/8 [00:08<00:00,  1.01s/it]


In [14]:
# Reassemble: join lines, split on sentence terminators only
arabic_continuous = " ".join(arabic_lines)
arabic_sentences = re.split(r'(?<=[.؟!])\s+', arabic_continuous)
arabic_sentences = [s.strip() for s in arabic_sentences if s.strip()]

print(f"{len(arabic_lines)} OCR lines -> {len(arabic_sentences)} sentences")
print(f"Min line confidence: {min(line_confidences):.4f}")

8 OCR lines -> 5 sentences
Min line confidence: 0.9569


In [15]:
# Save the handoff: sentences + confidence, to disk
handoff = {
    "sentences": arabic_sentences,
    "line_confidences": line_confidences,
    "min_confidence": min(line_confidences),
}
with open("../data/notebook11_handoff.json", "w", encoding="utf-8") as f:
    json.dump(handoff, f, ensure_ascii=False, indent=2)

print("Saved handoff to disk.")
for i, s in enumerate(arabic_sentences, 1):
    print(f"  {i}. {s}")

Saved handoff to disk.
  1. القانون هو مجموعة القواعد القانونية التي تهدف إلى تنظيم سلوك الأفراد في المجتمع وتكون مصحوبة بجزاء توقعه السلطة العامة على المخالف عند الاقتضاء.
  2. إذاً فالقانون علم اجتماعي، موضوعه الإنسان وسلوكه مع نظائره، أعماله وردود أفعاله، وهذا موضوع ضخم، متغير المضمون، غير معروف على وجه التحديد ويصعب عرضه بدقة في أغلب الأحوال، وهدفه حكم الجماعات الإنسانية، حتى لا تترك العلاقات بين الناس، عائلية أو اقتصادية أو سياسية، فوضى ينظمها كل فرد وفق رغبته ومشيئته، وإلا صدقت وتحققت مقولة الفيلسوف بسوت Bossuet: "حيث يملك الكل فعل ما يشاؤون لا يملك أحد فعل ما يشاء، وحيث لا سيد، فالكل سيد، وحيث الكل سيد فالكل عبيد".
  3. لذاكان لابد للمجتمع من نظام يحكم العلاقات بين الناس ويفرض الأمان في المجتمع.
  4. والقانون مصحوب دوماً بجزاء توقعه السلطة العامة على المخالف عند الاقتضاء حتى يكون التنظيم إجباري على سبيل الطلب الجازم بالأمر أو النهي وليس على سبيل المناشدة أو الرجاء.
  5. للقانون مباحث كثيرة باعتباره علما اجتماعيا بالدرجة الأولى، ولكنه جد صعب ومعقد، لذلك فإن ما يرد بشأنه من تعريفا

In [16]:
import gc

del recognition_predictor
del detection_predictor
del foundation_predictor
gc.collect()
torch.cuda.empty_cache()

print(f"GPU memory after clearing Surya: {torch.cuda.memory_allocated()/1e9:.2f} GB")

GPU memory after clearing Surya: 0.01 GB


In [17]:
import torch, json, time
from transformers import AutoProcessor, Gemma3ForConditionalGeneration

In [18]:
model_id = "google/translategemma-4b-it"
processor = AutoProcessor.from_pretrained(model_id)
model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id, dtype=torch.bfloat16, device_map="auto",
).eval()
print(f"TranslateGemma loaded. GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# Read the handoff from Phase 1
with open("../data/notebook11_handoff.json", "r", encoding="utf-8") as f:
    handoff = json.load(f)
arabic_sentences = handoff["sentences"]
print(f"Loaded {len(arabic_sentences)} sentences from Surya handoff.\n")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


TranslateGemma loaded. GPU: 6.35 GB
Loaded 5 sentences from Surya handoff.



In [19]:
print("=== Cascade: Surya -> reassembly -> TranslateGemma-text ===\n")
start_total = time.time()

for i, ar_sent in enumerate(arabic_sentences, 1):
    messages = [{"role": "user", "content": [{
        "type": "text",
        "source_lang_code": "ar",
        "target_lang_code": "de-DE",
        "text": ar_sent,
    }]}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        gen = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    de = processor.decode(gen[0][input_len:], skip_special_tokens=True).strip()

    print(f"Sentence {i}:")
    print(f"  AR: {ar_sent}")
    print(f"  DE: {de}")
    print()

print(f"Total translation time: {time.time() - start_total:.1f}s")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


=== Cascade: Surya -> reassembly -> TranslateGemma-text ===



Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Sentence 1:
  AR: القانون هو مجموعة القواعد القانونية التي تهدف إلى تنظيم سلوك الأفراد في المجتمع وتكون مصحوبة بجزاء توقعه السلطة العامة على المخالف عند الاقتضاء.
  DE: Das Gesetz ist eine Sammlung von rechtlichen Regeln, die darauf abzielen, das Verhalten von Einzelpersonen in der Gesellschaft zu regulieren. Es ist mit Sanktionen verbunden, die von der zuständigen Behörde bei Verstößen verhängt werden können.



Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Sentence 2:
  AR: إذاً فالقانون علم اجتماعي، موضوعه الإنسان وسلوكه مع نظائره، أعماله وردود أفعاله، وهذا موضوع ضخم، متغير المضمون، غير معروف على وجه التحديد ويصعب عرضه بدقة في أغلب الأحوال، وهدفه حكم الجماعات الإنسانية، حتى لا تترك العلاقات بين الناس، عائلية أو اقتصادية أو سياسية، فوضى ينظمها كل فرد وفق رغبته ومشيئته، وإلا صدقت وتحققت مقولة الفيلسوف بسوت Bossuet: "حيث يملك الكل فعل ما يشاؤون لا يملك أحد فعل ما يشاء، وحيث لا سيد، فالكل سيد، وحيث الكل سيد فالكل عبيد".
  DE: Da ist das Recht eine Sozialwissenschaft, deren Gegenstand der Mensch und sein Verhalten gegenüber anderen, seine Handlungen und Reaktionen sind. Dies ist ein sehr umfangreiches, sich ständig veränderndes und schwer präzise definierbares Thema. Ziel ist es, die Beziehungen zwischen Menschen – sei es im familiären, wirtschaftlichen oder politischen Bereich – zu regeln, damit keine Chaos entsteht, das von jedem Einzelnen nach seinen eigenen Wünschen und Überzeugungen bestimmt wird. Andernfalls würde die Aussage des Philo

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Sentence 3:
  AR: لذاكان لابد للمجتمع من نظام يحكم العلاقات بين الناس ويفرض الأمان في المجتمع.
  DE: Da war es für die Gesellschaft unerlässlich, ein System zu haben, das die Beziehungen zwischen den Menschen regelt und Sicherheit in der Gesellschaft gewährleistet.



Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Sentence 4:
  AR: والقانون مصحوب دوماً بجزاء توقعه السلطة العامة على المخالف عند الاقتضاء حتى يكون التنظيم إجباري على سبيل الطلب الجازم بالأمر أو النهي وليس على سبيل المناشدة أو الرجاء.
  DE: Das Gesetz ist immer mit einer von der zuständigen Behörde auferlegten Strafe verbunden, falls Verstöße vorliegen. Diese Strafe wird auf der Grundlage eines eindeutigen Befehls oder einer Verbots auferlegt, und nicht als Bitte oder Ermahnung.

Sentence 5:
  AR: للقانون مباحث كثيرة باعتباره علما اجتماعيا بالدرجة الأولى، ولكنه جد صعب ومعقد، لذلك فإن ما يرد بشأنه من تعريفات مرن جداً، ويتضمن عدداً من وجهات النظر والاستثناءات وذلك على خلاف العلوم الرياضية: كالفيزياء والكيمياء، وفي أغلب الأحيان يختلف القانون في مضامينه من مجتمع لآخر حسب الإرث التاريخي لكل مجتمع ومن أبرز فروع القانون نجد القانون العام والقانون الخاص.
  DE: Das Recht ist ein sehr komplexes und anspruchsvolles Fach, das vor allem als Sozialwissenschaft betrachtet wird. Daher sind die Definitionen sehr flexibel und umfassen verschiedene Per